# 6주차 · 신경망으로 불량 예측하고 검증하기

**이번 실습:** 4주차와 같은 공정 표에 작은 신경망을 적용하고, 검증 자료로 규제 세기를 고릅니다.  
**준비:** 4주차 분류와 5주차 뉴런의 학습 원리

이 파일 하나를 위에서부터 순서대로 실행합니다. 주차 안의 이론 차시마다 별도 코드를 실행할 필요는 없습니다. 오른쪽 위 커널은 **Python (semi-ai)**로 고릅니다. `#` 뒤의 한국어 주석을 한 줄씩 읽고 실행해 주세요. 데이터는 교육용 합성 자료입니다.

## 0. 실습 폴더 준비

VS Code와 Colab 어디서 열어도 `data` 폴더를 찾도록 실습 폴더 위치를 정합니다. Colab에서는 처음 한 번 실습 자료를 내려받습니다.

In [ ]:
import sys, subprocess  # 실행 환경 확인과 명령 실행 도구입니다.
from pathlib import Path  # 파일 위치 도구입니다.
if "google.colab" in sys.modules:  # Colab에서 실행 중이라면
    folder = Path("/content/semiconductor_ai_beginner")  # 실습 자료를 둘 위치입니다.
    if not folder.exists():  # 아직 내려받지 않았다면
        subprocess.run(["git", "clone", "-q", "https://github.com/SeokJH-TAs/semiconductor_ai_beginner.git", str(folder)], check=True)  # 실습 자료 전체를 내려받습니다.
else:  # VS Code 등 내 컴퓨터에서 실행 중이라면
    folder = Path.cwd()  # 현재 폴더입니다.
    if folder.name == "notebooks":  # 노트북 폴더 안이라면
        folder = folder.parent  # 실습 전체 폴더로 올라갑니다.
print(folder)  # 실습 폴더 위치를 확인합니다.

## 1. 입력과 정답을 세 묶음으로 나누기

입력은 공정 중 알 수 있는 네 열, 정답은 공정 후 불량 라벨입니다. 학습 60%, 검증 20%, 최종 시험 20%로 나눕니다. 같은 자료를 쓰되 트리 대신 은닉층이 있는 모델을 사용합니다.

In [1]:
from pathlib import Path  # 파일 위치를 다루는 도구입니다.
import pandas as pd  # 표를 읽는 도구입니다.
from sklearn.model_selection import train_test_split  # 자료를 나누는 도구입니다.
data = pd.read_csv(folder / "data" / "process.csv")  # 같은 공정 표를 읽습니다.
X = data[["temperature_c", "pressure_torr", "gas_flow_sccm", "rf_power_w"]]  # 사전 공정 조건입니다.
y = data["defect"]  # 공정 후 불량 정답입니다.
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)  # 시험 20%를 남깁니다.
X_train, X_valid, y_train, y_valid = train_test_split(X_dev, y_dev, test_size=0.25, random_state=42, stratify=y_dev)  # 개발 집합을 다시 나눕니다.
print(len(X_train), len(X_valid), len(X_test))  # 144, 48, 48인지 확인합니다.

144 48 48


## 2. 작은 신경망 두 후보 준비하기

은닉층 하나에 뉴런 8개를 둡니다. `(8,)`는 뉴런 수 설정 묶음입니다. `activation="tanh"`는 중간 출력을 -1~1로 바꾸는 함수이고 `solver="lbfgs"`는 작은 데이터의 손실을 줄이는 최적화 도구 이름입니다. `alpha`는 가중치 크기 벌점의 세기입니다. 두 후보는 이 벌점만 다릅니다.

`make_pipeline`은 표준화 → 모델 순서로 연결합니다. 학습 자료로만 표준화 기준을 배우고 검증·시험에는 그 기준을 그대로 적용합니다.

In [2]:
from sklearn.pipeline import make_pipeline  # 도구를 순서대로 연결합니다.
from sklearn.preprocessing import StandardScaler  # 입력 크기를 맞추는 도구입니다.
from sklearn.neural_network import MLPClassifier  # 은닉층이 있는 분류 신경망입니다.
weak_network = MLPClassifier(hidden_layer_sizes=(8,), activation="tanh", solver="lbfgs", alpha=0.01, max_iter=3000, random_state=42)  # 약한 벌점 후보입니다.
strong_network = MLPClassifier(hidden_layer_sizes=(8,), activation="tanh", solver="lbfgs", alpha=1.0, max_iter=3000, random_state=42)  # 더 큰 벌점 후보입니다.
weak = make_pipeline(StandardScaler(), weak_network)  # 표준화 뒤 첫 신경망을 연결합니다.
strong = make_pipeline(StandardScaler(), strong_network)  # 같은 순서로 둘째 후보를 만듭니다.
weak.fit(X_train, y_train)  # 학습 묶음으로 첫 후보를 배웁니다.
strong.fit(X_train, y_train)  # 같은 묶음으로 둘째 후보도 배웁니다.
print("두 후보 학습 완료")  # 다음 구역으로 넘어갈 준비가 됐습니다.

두 후보 학습 완료


## 3. 학습과 검증 손실을 비교하기

교차 엔트로피는 맞는 정답에 확률을 얼마나 주었는지 채점합니다. 작을수록 좋습니다. 학습 손실만 낮고 검증 손실이 높으면 외웠을 수 있습니다. 어떤 후보를 고를지는 검증 손실로 결정합니다.

In [3]:
from sklearn.metrics import log_loss  # 확률 예측의 교차 엔트로피를 계산합니다.
weak_train = log_loss(y_train, weak.predict_proba(X_train))  # 첫 후보의 학습 손실입니다.
weak_valid = log_loss(y_valid, weak.predict_proba(X_valid))  # 첫 후보의 검증 손실입니다.
strong_train = log_loss(y_train, strong.predict_proba(X_train))  # 둘째 후보의 학습 손실입니다.
strong_valid = log_loss(y_valid, strong.predict_proba(X_valid))  # 둘째 후보의 검증 손실입니다.
print("약한 규제: 학습 / 검증", weak_train, weak_valid)  # 첫 후보의 두 점수를 봅니다.
print("강한 규제: 학습 / 검증", strong_train, strong_valid)  # 둘째 후보도 봅니다.

약한 규제: 학습 / 검증 0.032254734233782535 0.8894056597970349
강한 규제: 학습 / 검증 0.23878619397673376 0.29101485937636357


## 4. 선택을 끝내고 최종 시험 한 번 보기

설정을 선택한 뒤 개발 자료 전체로 다시 학습하고 남겨 둔 시험을 한 번 확인합니다. 최종 시험 점수를 보고 다시 규제를 고르지 않습니다. 이 실습의 자료는 독립 합성 행이므로 무작위 분할하며 실제 로트·시간 자료는 그 구조에 맞는 분할이 필요합니다.

In [4]:
from sklearn.metrics import confusion_matrix, recall_score  # 놓친 불량을 세는 도구입니다.
chosen = strong  # 우선 강한 벌점 후보를 선택합니다.
if weak_valid < strong_valid:  # 검증 손실이 첫 후보에서 더 작았다면
    chosen = weak  # 첫 후보로 선택을 바꿉니다.
chosen.fit(X_dev, y_dev)  # 설정 선택 후 개발 자료 전체로 다시 학습합니다.
probability = chosen.predict_proba(X_test)  # 시험 입력에 대한 확률을 구합니다.
predicted = chosen.predict(X_test)  # 같은 입력의 0 또는 1 판정을 구합니다.
print(log_loss(y_test, probability))  # 최종 시험 손실입니다.
print(confusion_matrix(y_test, predicted, labels=[0, 1]))  # 실제 행·예측 열의 네 칸 표입니다.
print(recall_score(y_test, predicted))  # 실제 불량 중 찾은 비율입니다.

0.254140438661328
[[34  2]
 [ 4  8]]
0.6666666666666666


## 말로 정리하기

입력 4개 → 은닉 뉴런 8개 → 불량 판정이라는 구조와 학습·검증·시험의 역할을 설명해 보세요. 4주차 트리와 신경망을 비교할 때는 같은 시험 집합과 같은 지표를 사용해야 공정합니다. 두 주차의 분할이 다르므로 저장된 점수만 보고 우열을 결론내리지 않습니다.

드롭아웃·배치 정규화·Adam·XOR·깊은 신경망의 모든 이론을 각각 별도 코딩하는 과제는 없습니다.